In [1]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# Import the database access layer for interacting with MongoDB
from crud import AnimalShelter

# Implement the module query layer
from query_service import build_query, validate_filter

###########################
# Data Manipulation / Model
###########################


# Connect to database via CRUD Module
db = AnimalShelter()


# Load all animal records into a DataFrame for the initial dashboard view
df = pd.DataFrame.from_records(db.read({}))

# Remove the MongoDB ObjectId field so the Dash table can render the data safely
df.drop(columns=['_id'],inplace=True)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

# Load dashboard logo image
image_filename = 'grazioso_logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height':'100px'}),
    html.H4(('Dashboard by Dante Nardulli - Capstone'), style={'textAlign': 'center'}),
    html.Center(html.B(html.H1('CS-340/499 Dashboard'))),
    html.Hr(),
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All Dogs', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='all',
            labelStyle={'display': 'inline-block', 'margin': '10px'}
        )


    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         column_selectable="single",
                         row_selectable="single",
                         row_deletable=False,
                         selected_rows=[0],  # Default selection to first row
                         page_action='native',
                         page_current=0,
                         page_size=10,  # Show 10 records per page
                         style_table={'overflowX': 'auto'},
                         style_cell={
                             'minWidth': '150px', 'width': '150px', 'maxWidth': '150px',
                             'whiteSpace': 'normal'
                        },

                    ),

                        
    html.Br(),
    html.Hr(),
# Display the chart and map side by side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])


#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    #Update the data table based on the selected rescue filter.
    valid_filter = validate_filter(filter_type)
    query = build_query(valid_filter)
    df = pd.DataFrame.from_records(db.read(query))
    df.drop(columns=['_id'], inplace=True, errors='ignore')
    return df.to_dict('records')

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if not viewData:
        return html.Div("No data available to generate chart.")
    
    try:
        dff = pd.DataFrame(viewData)
        if 'breed' not in dff.columns:
            return html.Div("Column 'breed' not found in data.")
        fig = px.pie(dff, names='breed', title='Breed Distribution')
        return dcc.Graph(figure=fig)
    except Exception as e:
        return html.Div(f"Error generating graph: {str(e)}")
    
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': { 'column_id': i },
        'backgroundColor': "#DAD2FF"
    } for i in selected_columns]


# Update the map based on the currently visible table data and selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Center the map on Austin, Texas
    # Add a marker for the selected animal record
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run(debug=True, host="127.0.0.1", port=8050, jupyter_mode="external")


Dash app running on http://127.0.0.1:8050/
